# 00 — Random-attribution baseline (the CMI floor)

Calibrates the CMI scale. The ladder's CMI values (0.387–0.654) currently have **no floor reference** — a
reader cannot tell whether 0.39 is good. This notebook generates **random attribution vectors** over the 50
regions and pushes them through the **existing** deletion-curve + CMI machinery — **no attribution method is
run**; random vectors simply replace the attributions. Everything else is identical to the main runs, so the
floor is directly comparable.

Two things it settles:
- **The floor** for the central grid: how much of each method's CMI is signal above chance ordering.
- **PES behaviour at chance**: a random ranking should get the deletion direction right ~half the time, so
  random PES should be ~0.5. If random PES comes out much higher, PES saturates even for meaningless
  attributions — a finding about the metric (its ~1.0 values across the ladder would carry even less
  information than currently thought).

> **You run this.** §2 is the compute (~16 min total; per-model estimates below). §1, §3–§5 are seconds.

## §1. Setup, settings, loaders (all parameters once)

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
import torch
import sleep_edf.config as cfg
from sleep_edf.loader import load_sleep_edf
from harness.models.cnn import build_cnn, torch_predict_proba
from harness.models.transformer import build_transformer
from harness.xai.regions import build_region_grid
from harness.xai.deletion_curves import perturbation_curves      # EXISTING machinery (25 steps, MoRF/LeRF)
from harness.xai.cmi import compute_cmi                          # EXISTING (DDS -> PES -> CMI)
from harness.xai.feature_ablation import feature_ablation        # only for the §5 oracle check
from harness.xai.concentration import region_reliance            # only for the §5 oracle check

# ── Settings — identical to the main XAI runs so the floor is comparable ──────
SEEDS      = [0, 1, 2, 3, 4]     # the same 5 MODEL seeds; aggregated mean ± std
PM         = "zero"             # perturbation method
N_EVAL     = 500               # the fixed eval subset (loaded, never regenerated)
DEV        = "cpu"             # deletion curves = tiny batch-1 forwards -> CPU fastest
# target class = PREDICTED (perturbation_curves target_class=None -> argmax on the unperturbed signal)

# ── Random-attribution seed: explicit, reproducible, INDEPENDENT of the model seeds ──
RANDOM_SEED = 20260816
# One RandomState, consumed in a FIXED nested order (rung 2,3,4,5 -> seed 0..4 -> sample 0..499),
# drawing a FRESH rand(50) per (model, model-seed, sample). Fixed order => fully reproducible; fresh per
# sample => the per-sample variance is not understated.

MET = PROJECT_ROOT / "sleep_edf" / "results" / "metrics"
CKPT = PROJECT_ROOT / "sleep_edf" / "results" / "checkpoints"
grid = build_region_grid(cfg.INPUT_LENGTH, cfg.REGION_SIZE_PRIMARY_PCT)   # 50 regions of 60

RUNGS = {2: ("shallow", "model2_shallow_cnn"), 3: ("medium", "model3_medium_cnn"),
         4: ("deep", "model4_deep_cnn"), 5: ("transformer", "model5_transformer")}
def load_model(rung, seed, device=DEV):
    variant, name = RUNGS[rung]
    if variant == "transformer":
        m = build_transformer(input_length=cfg.INPUT_LENGTH, in_channels=cfg.IN_CHANNELS,
                              n_classes=cfg.N_CLASSES, patch_size=cfg.TRANSFORMER_PATCH_SIZE)
    else:
        m = build_cnn(variant, n_classes=cfg.N_CLASSES, in_channels=cfg.IN_CHANNELS, kernel_size=cfg.CNN_KERNEL_SIZE)
    m.load_state_dict(torch.load(CKPT / f"{name}_seed{seed}.pt", map_location="cpu")); m.eval()
    return m.to(device)

# The SAME fixed 500-sample eval subset the ladder used (loaded, never regenerated).
EVAL_IDX_PATH = MET / "xai_eval_subset_idx.npy"
if not EVAL_IDX_PATH.exists():
    raise FileNotFoundError(f"shared eval subset {EVAL_IDX_PATH} not found — run the Model 2 XAI notebook's §2 first.")
X_test, y_test = load_sleep_edf("test", verbose=False)
eval_idx = np.load(EVAL_IDX_PATH); eval_sigs = X_test[eval_idx].astype(float)
assert len(eval_idx) == N_EVAL
print(f"grid {grid.n_regions} regions | N_EVAL {len(eval_idx)} (shared subset) | RANDOM_SEED {RANDOM_SEED}")
print("per-model est: Model 2 ~1.5 min, Model 3 ~2 min, Model 4 ~6 min, Model 5 ~7 min  (~16 min total)")

## §2. ▶ Compute the random floor  (~16 min total, CPU, saves per model)

For each model and each of its 5 seeds, draw a fresh random vector per sample and push it through
`perturbation_curves` (zero PM, predicted class, 25 steps, MoRF/LeRF) → `compute_cmi`. Aggregated to mean ±
std across seeds. One `RandomState(RANDOM_SEED)` is consumed in the fixed order rung→seed→sample, so the floor
is reproducible. Results saved to `random_floor_results.json` (per model, after each model completes).

In [ ]:
rng = np.random.RandomState(RANDOM_SEED)     # consumed in fixed order rung -> seed -> sample
floor = {}
for rung in [2, 3, 4, 5]:
    per_seed = []
    t_model = time.perf_counter()
    for si, seed in enumerate(SEEDS):
        pp = torch_predict_proba(load_model(rung, seed), device=DEV)
        M, L = [], []
        t0 = time.perf_counter()
        for j in range(N_EVAL):
            rv = rng.rand(grid.n_regions)        # FRESH random attribution per (rung, seed, sample)
            c = perturbation_curves(pp, eval_sigs[j], grid, rv, method=PM)   # target None -> predicted class
            M.append(c["MoRF"]); L.append(c["LeRF"])
        r = compute_cmi(M, L)
        per_seed.append({"seed": seed, "CMI": r["CMI"], "DDS": r["DDS"], "PES": r["PES"]})
        if si == 0:
            dt = time.perf_counter() - t0
            print(f"  Model {rung} seed {seed}: {dt:.0f}s -> est ~{dt*len(SEEDS)/60:.1f} min for this model", flush=True)
    A = lambda k: (float(np.mean([d[k] for d in per_seed])), float(np.std([d[k] for d in per_seed])))
    cmi, dds, pes = A("CMI"), A("DDS"), A("PES")
    floor[rung] = {"CMI_mean": cmi[0], "CMI_std": cmi[1], "DDS_mean": dds[0], "DDS_std": dds[1],
                   "PES_mean": pes[0], "PES_std": pes[1], "per_seed": per_seed}
    print(f"Model {rung} DONE ({(time.perf_counter()-t_model)/60:.1f} min): "
          f"CMI {cmi[0]:.3f}±{cmi[1]:.3f} | DDS {dds[0]:.3f}±{dds[1]:.3f} | PES {pes[0]:.3f}±{pes[1]:.3f}", flush=True)
    # save after each model (partial results survive an interruption)
    json.dump({"random_seed": RANDOM_SEED, "n_eval": N_EVAL, "pm": PM, "seeds": SEEDS, "floor": floor},
              open(MET / "random_floor_results.json", "w"), indent=2, default=float)
print("\nsaved:", (MET / "random_floor_results.json").name)

## §3. PES at chance — is the consistency metric behaving?

**Note on scale.** PES is defined as *fraction-positive − fraction-negative* of the per-sample DDS. A random
ranking gets the deletion direction right about half the time (fraction-positive ≈ 0.5), which makes
**PES ≈ 0** — *not* 0.5. (The "half the time" intuition is the fraction-positive = (1 + PES)/2, reported
below.) So the question is whether random PES sits near **0**:

- **random PES ≈ 0** (fraction-positive ≈ 0.5) → PES is a genuine sign-consistency measure that DISCRIMINATES;
  the ladder's ~1.0 (fraction-positive ≈ 1.0) reflects real consistency, not saturation on garbage. Reassuring.
- **random PES well ABOVE 0** (fraction-positive ≫ 0.5) → the metric is POSITIVELY BIASED even for meaningless
  attributions: the ~1.0 across the ladder would then carry even less information than the saturation finding
  implies (CMI effectively |DDS| only). Flag prominently.

In [ ]:
print(f"{'model':<10}{'random PES (mean±std)':>24}{'frac-positive (1+PES)/2':>26}")
for rung in [2, 3, 4, 5]:
    f = floor[rung]; pes_str = f"{f['PES_mean']:.3f}±{f['PES_std']:.3f}"; fp = (1 + f['PES_mean']) / 2
    print(f"Model {rung:<4}{pes_str:>24}{fp:>26.3f}")
mean_pes = np.mean([floor[r]["PES_mean"] for r in [2,3,4,5]])
print(f"\nmean random PES across rungs: {mean_pes:.3f}  (fraction-positive {(1+mean_pes)/2:.3f}; chance = 0.5)")
if abs(mean_pes) <= 0.2:
    print(">>> Random PES ~0 (fraction-positive ~0.5) as expected -> PES DISCRIMINATES; the ladder's ~1.0 is "
          "real sign-consistency, not saturation on garbage.")
else:
    print(">>> FINDING: random PES is well away from 0 -> the metric is BIASED even for meaningless attributions. "
          "The ~1.0 PES across the ladder then carries even less information than the saturation finding implied; "
          "CMI is effectively |DDS| only. Flag this prominently.")

## §4. Floor vs the real methods — margin above chance

Each rung's random floor next to its FA, KS, IG CMI (read from the saved XAI results JSONs — not hardcoded),
with the margin above floor. Then Model 5's **attention** (last-layer 0.058, rollout 0.078) against Model 5's
floor — labelled separately, since attention is not comparable to the other three.

In [ ]:
xai = {r: json.load(open(MET / f"model{r}_xai_cmi_results.json")) for r in [2,3,4,5]}
print(f"{'model':<9}{'RANDOM floor':>14}{'FA':>9}{'(marg)':>8}{'KS':>9}{'(marg)':>8}{'IG':>9}{'(marg)':>8}")
for r in [2,3,4,5]:
    fl = floor[r]["CMI_mean"]; m = xai[r]["methods"]
    fa,ks,ig = m["FeatureAblation"]["CMI_mean"], m["KernelSHAP"]["CMI_mean"], m["IntegratedGradients"]["CMI_mean"]
    print(f"Model {r:<3}{fl:>14.3f}{fa:>9.3f}{fa-fl:>+8.3f}{ks:>9.3f}{ks-fl:>+8.3f}{ig:>9.3f}{ig-fl:>+8.3f}")
print("\nAll three shared methods sit well ABOVE the random floor -> they carry real ordering signal.")

# Attention vs the Model 5 floor (SEPARATE — attention is not comparable to FA/KS/IG).
fl5 = floor[5]["CMI_mean"]
att = xai[5]["methods"]["Attention"]["CMI_mean"]
roll = json.load(open(MET / "model5_xai_rollout_results.json"))["CMI_mean"]
print(f"\n[SEPARATE — attention, not comparable] Model 5 random floor {fl5:.3f}:")
print(f"  last-layer attention CMI {att:.3f}  (margin {att-fl5:+.3f})  <- is it AT the floor?")
print(f"  rollout attention   CMI {roll:.3f}  (margin {roll-fl5:+.3f})")
print("  If last-layer sits AT the random floor, that is the sharpest statement of its degeneracy.")

## §5. Oracle verification — FeatureAblation(zero) == single-region deletion reliance?

The project frames KS and IG as "how close to the ORACLE", where the oracle is FeatureAblation(zero). This
cell verifies that FA(zero) really equals the model's single-region-deletion reliance (the quantity the
concentration measure uses) — on a few samples, one model — so the framing is demonstrated, not asserted. This
is NOT the expensive cumulative/greedy oracle (out of scope); it is a cheap per-region equality check.

In [ ]:
pp = torch_predict_proba(load_model(2, 0), device=DEV)   # one model, a few samples
maxd = 0.0
for i in eval_idx[:6]:
    s = X_test[i].astype(float)
    fa = feature_ablation(pp, s, grid, "zero")     # Captum grouped FA: (P_full - P_ablated), predicted class
    rr = region_reliance(pp, s, grid, "zero")      # hand-rolled single-region deletion, predicted class
    maxd = max(maxd, float(np.max(np.abs(fa - rr))))
print(f"max|FA(zero) - single-region reliance| over 6 samples (Model 2 seed 0): {maxd:.3e}")
print(">>>", "EXACT — the oracle equivalence HOLDS on Sleep-EDF; 'how close to the oracle' is demonstrated."
      if maxd < 1e-6 else ("APPROXIMATE (max|Δ| %.2e)" % maxd if maxd < 1e-3 else "DOES NOT HOLD — do not use the oracle framing."))